# 04. RII 전처리 노트북
## 강우·수해 취약도 지수 (Rain/Flood Isolation Index)

**프로젝트**: 비가 오면 누가 고립되는가 — 서울시 고령 1인가구의 복합위기 취약지역 탐지  
**분석 단위**: 서울시 행정동  
**대상 연도**: 2019–2021, 최종 병합용 2021년 파일 별도 저장

이번 버전은 공간 결합 실패를 조용히 0으로 처리하지 않습니다. `geopandas`와 행정동 경계 파일이 없으면 중단하고, TRACE/홍수예측 SHP와 행정동 경계의 CRS, bounds, geometry type, 행 개수를 출력해 원인을 확인합니다.

---
## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

try:
    import geopandas as gpd
except ImportError as e:
    raise ImportError(
        "RII 전처리는 TRACE/홍수예측 SHP와 행정동 경계의 공간 결합이 핵심이므로 geopandas가 반드시 필요합니다. "
        "설치 예시: conda install -c conda-forge geopandas"
    ) from e

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)

---
## 2. 경로 설정

`ADMIN_BOUNDARY_PATH`, `ADMIN_CODE_COL`, `ADMIN_GU_COL`, `ADMIN_DONG_COL`은 필요할 때 직접 지정할 수 있습니다.

In [ ]:
# 프로젝트 루트
BASE_DIR = Path("c:/Tsum2026/T_SUM2026")

# 원본 데이터 폴더
RAW_RII_DIR = BASE_DIR / "data" / "raw" / "RII_rain_flood"
FLOOD_PRED_DIR = RAW_RII_DIR / "flood_prediction"
FLOOD_TRACE_DIR = RAW_RII_DIR / "flood_trace"

# 강수량 CSV 후보 경로
RAINFALL_CANDIDATES = [
    RAW_RII_DIR / "서울시_강수량_2019_2021.csv",
    RAW_RII_DIR / "rainfall" / "서울시_강수량_2019_2021.csv",
    RAW_RII_DIR / "서울시_강수량_2019_2021" / "서울시_강수량_2019_2021.csv",
]
RAINFALL_PATH = next((p for p in RAINFALL_CANDIDATES if p.exists()), RAINFALL_CANDIDATES[0])

# 행정동 경계 후보 폴더
ADMIN_BOUNDARY_DIRS = [
    BASE_DIR / "data" / "external" / "seoul_admin_dong_boundary",
    BASE_DIR / "data" / "raw" / "boundary",
]

# 행정동 경계 파일/컬럼 직접 지정 옵션
# 예: ADMIN_BOUNDARY_PATH = BASE_DIR / "data" / "external" / "seoul_admin_dong_boundary" / "seoul_hdong.shp"
ADMIN_BOUNDARY_PATH = None
ADMIN_CODE_COL = None   # 예: "ADM_CD"
ADMIN_GU_COL = None     # 예: "SGG_NM"
ADMIN_DONG_COL = None   # 예: "ADM_NM"

# 저장 경로
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_ALL_CSV = PROCESSED_DIR / "rii_by_dong_2019_2021.csv"
OUT_2021_CSV = PROCESSED_DIR / "rii_by_dong_2021.csv"
OUT_2021_GPKG = PROCESSED_DIR / "rii_by_dong_2021.gpkg"

TARGET_CRS = "EPSG:5179"
TARGET_YEARS = [2019, 2020, 2021]

for label, path in [
    ("RII 원본 폴더", RAW_RII_DIR),
    ("강수량 CSV", RAINFALL_PATH),
    ("홍수 예측 폴더", FLOOD_PRED_DIR),
    ("침수흔적 폴더", FLOOD_TRACE_DIR),
    ("전처리 저장 폴더", PROCESSED_DIR),
]:
    mark = "✓" if path.exists() else "✗ 없음"
    print(f"[{mark}] {label}: {path}")

print("\n행정동 경계 후보 폴더")
for path in ADMIN_BOUNDARY_DIRS:
    mark = "✓" if path.exists() else "✗ 없음"
    print(f"[{mark}] {path}")

---
## 3. 공통 함수 정의

In [ ]:
def to_numeric_safe(series):
    '''쉼표, 공백, '-', 결측을 정리한 뒤 숫자로 변환합니다.'''
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"-": "0", "": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce",
    )


def minmax_scale(series):
    '''Min-Max Scaling. 모든 값이 같으면 변별력이 없으므로 0으로 둡니다.'''
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


def first_existing(columns, candidates):
    '''후보 컬럼 중 실제 존재하는 첫 컬럼명을 반환합니다.'''
    for col in candidates:
        if col in columns:
            return col
    return None


def clean_name(series):
    '''행정구역명 공백과 따옴표를 정리합니다.'''
    return series.astype(str).str.strip().str.strip('"').replace({"nan": np.nan, "None": np.nan})


def make_dong_key(df, gu_col="자치구명", dong_col="행정동명"):
    '''행정동명 단독 병합은 중복 위험이 있으므로 자치구명과 행정동명을 함께 묶습니다.'''
    if gu_col in df.columns and dong_col in df.columns:
        return clean_name(df[gu_col]) + "_" + clean_name(df[dong_col])
    return pd.Series(np.nan, index=df.index)


def infer_missing_crs_from_bounds(gdf, default_crs=TARGET_CRS):
    '''CRS가 없는 SHP의 bounds를 보고 가능한 원 CRS를 추정합니다.'''
    if gdf is None or gdf.empty:
        return default_crs
    minx, miny, maxx, maxy = gdf.total_bounds
    # DS_FLOODING 파일은 x=18~21만, y=53~56만대라 EPSG:5186(중부원점) 좌표로 판단됩니다.
    if 100000 <= minx <= 300000 and 400000 <= miny <= 700000:
        return "EPSG:5186"
    # TRACE 파일은 x=93~97만, y=194~196만대라 EPSG:5179 좌표로 판단됩니다.
    if 900000 <= minx <= 1100000 and 1800000 <= miny <= 2100000:
        return "EPSG:5179"
    return default_crs


def ensure_projected_crs(gdf, target_crs=TARGET_CRS, source_name="gdf"):
    '''공간 결합 전 좌표계를 통일합니다. CRS가 없으면 bounds로 원 CRS를 추정합니다.'''
    if gdf is None or gdf.empty:
        return gdf
    if gdf.crs is None:
        inferred_crs = infer_missing_crs_from_bounds(gdf, default_crs=target_crs)
        print(f"[주의] {source_name}: CRS가 없어 bounds 기준으로 {inferred_crs}로 가정합니다. 실제 메타데이터 확인 필요")
        gdf = gdf.set_crs(inferred_crs, allow_override=True)
    return gdf.to_crs(target_crs)


def print_gdf_diagnostics(gdf, name):
    '''공간 결합 문제를 찾기 위해 CRS, bounds, geometry type, 행 개수를 출력합니다.'''
    print(f"\n=== {name} 공간 진단 ===")
    if gdf is None:
        print("GeoDataFrame is None")
        return
    print(f"행 개수: {len(gdf):,}")
    print(f"CRS: {gdf.crs}")
    if len(gdf) > 0:
        print(f"bounds [minx, miny, maxx, maxy]: {gdf.total_bounds}")
        print("geometry type:")
        print(gdf.geom_type.value_counts(dropna=False))
        print("빈 geometry 수:", int(gdf.geometry.is_empty.sum()))
        print("결측 geometry 수:", int(gdf.geometry.isna().sum()))


def bounds_overlap(gdf1, gdf2):
    '''두 GeoDataFrame의 전체 bounds가 겹치는지 확인합니다.'''
    if gdf1 is None or gdf2 is None or gdf1.empty or gdf2.empty:
        return False
    a = gdf1.total_bounds
    b = gdf2.total_bounds
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def diagnose_join_failure(source_gdf, admin_gdf, source_name):
    '''공간 결합이 0건일 때 CRS/bounds/geometry 상태를 비교해 원인을 추정합니다.'''
    print(f"\n[경고] {source_name} 공간 결합 매칭이 0건입니다. 원인 진단을 출력합니다.")
    print_gdf_diagnostics(source_gdf, f"{source_name} 원본")
    print_gdf_diagnostics(admin_gdf, "행정동 경계")
    print("bounds 겹침 여부:", bounds_overlap(source_gdf, admin_gdf))
    print("가능한 원인:")
    print("1) 두 데이터의 실제 CRS가 다르지만 CRS 메타데이터가 잘못 지정됨")
    print("2) DS_FLOODING 또는 TRACE의 CRS가 EPSG:5179가 아닌데 .prj가 없어 EPSG:5179로 잘못 가정됨")
    print("3) 행정동 경계가 서울 행정동 경계가 아니거나 다른 행정구역 단위임")
    print("4) geometry가 손상되었거나 빈 geometry가 포함됨")


def standardize_admin_columns(gdf, code_col=None, gu_col=None, dong_col=None):
    '''행정동 경계 데이터의 주요 컬럼을 표준 이름으로 맞춥니다. 직접 지정값이 있으면 우선 사용합니다.'''
    if gdf is None or gdf.empty:
        return gdf

    print("행정동 경계 원본 컬럼:")
    print(list(gdf.columns))

    code_col = code_col or first_existing(gdf.columns, [
        "행정동코드", "ADM_CD", "ADM_DR_CD", "H_DNG_CD", "HDONG_CD", "ADSTRD_CD",
        "ADSTRD_CODE", "DONG_CD", "dong_cd", "adm_cd", "adm_cd2", "행정기관코드"
    ])
    gu_col = gu_col or first_existing(gdf.columns, [
        "자치구명", "SIG_KOR_NM", "GU_NAM", "GU_NAME", "SGG_NM", "시군구명", "구명", "gu", "SIGUNGU_NM", "sggnm"
    ])
    dong_col = dong_col or first_existing(gdf.columns, [
        "행정동명", "ADM_NM", "ADM_DR_NM", "H_DNG_NM", "HDONG_NM", "ADSTRD_NM", "동명", "dong", "EMD_KOR_NM", "adm_nm"
    ])

    print("선택된 행정동 경계 컬럼:")
    print({"행정동코드": code_col, "자치구명": gu_col, "행정동명": dong_col})

    if dong_col is None:
        raise ValueError("행정동명 후보 컬럼을 찾지 못했습니다. ADMIN_DONG_COL에 실제 컬럼명을 지정하세요.")

    rename_map = {}
    if code_col and code_col != "행정동코드":
        rename_map[code_col] = "행정동코드"
    if gu_col and gu_col != "자치구명":
        rename_map[gu_col] = "자치구명"
    if dong_col and dong_col != "행정동명":
        rename_map[dong_col] = "행정동명"
    gdf = gdf.rename(columns=rename_map).copy()

    if "행정동명" in gdf.columns:
        gdf["행정동명"] = clean_name(gdf["행정동명"])
    if "자치구명" in gdf.columns:
        gdf["자치구명"] = clean_name(gdf["자치구명"])
    else:
        print("[주의] 자치구명 컬럼이 없습니다. 행정동코드가 없으면 중복 행정동명 문제가 생길 수 있습니다.")
        gdf["자치구명"] = np.nan
    if "행정동코드" in gdf.columns:
        gdf["행정동코드"] = clean_name(gdf["행정동코드"])
    else:
        print("[주의] 행정동코드 컬럼이 없습니다. 후속 병합은 자치구명+행정동명 기준입니다.")
        gdf["행정동코드"] = np.nan

    # raqoon886/Local_HangJeongDong 계열은 adm_nm이
    # "서울특별시 종로구 사직동"처럼 전체 행정구역명으로 들어옵니다.
    # 최종 결과의 행정동명은 마지막 토큰만 사용하고, 자치구명은 sggnm을 우선 사용합니다.
    if "adm_nm" in gdf.columns:
        adm_parts = clean_name(gdf["adm_nm"]).str.split()
        if "sggnm" in gdf.columns:
            gdf["자치구명"] = clean_name(gdf["sggnm"])
        elif gdf["자치구명"].isna().all():
            gdf["자치구명"] = adm_parts.str[-2]
        gdf["행정동명"] = adm_parts.str[-1]

    # adm_nm이 이미 행정동명으로 rename된 경우도 처리합니다.
    full_name_mask = gdf["행정동명"].astype(str).str.contains(" ", regex=False)
    if full_name_mask.any():
        parts = gdf.loc[full_name_mask, "행정동명"].astype(str).str.split()
        if "자치구명" in gdf.columns:
            gdf.loc[full_name_mask & gdf["자치구명"].isna(), "자치구명"] = parts.str[-2]
        gdf.loc[full_name_mask, "행정동명"] = parts.str[-1]

    gdf["dong_key"] = make_dong_key(gdf)
    return gdf


def calculate_rank_grade_by_year(df, score_col="RII"):
    '''RII 순위와 등급을 기준연도 내부에서만 계산합니다.'''
    out = df.copy()
    out["RII_rank"] = out.groupby("기준연도")[score_col].rank(ascending=False, method="min").astype(int)

    parts = []
    for year, part in out.groupby("기준연도", dropna=False):
        part = part.copy()
        if part[score_col].nunique(dropna=False) <= 1:
            print(f"[경고] {year}년 RII 값이 모두 동일하여 RII_grade를 3으로 부여합니다.")
            part["RII_grade"] = 3
        else:
            try:
                part["RII_grade"] = pd.qcut(
                    part[score_col].rank(method="first"),
                    q=5,
                    labels=[1, 2, 3, 4, 5],
                ).astype(int)
            except ValueError:
                part["RII_grade"] = pd.qcut(
                    part[score_col].rank(method="first"),
                    q=5,
                    labels=[1, 2, 3, 4, 5],
                    duplicates="drop",
                )
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

---
## 4. 강수량 데이터 전처리

In [ ]:
print("=== 강수량 파일 앞부분 확인 ===")
if RAINFALL_PATH.exists():
    with open(RAINFALL_PATH, encoding="cp949") as f:
        for i in range(15):
            print(i, f.readline().rstrip())
else:
    print(f"강수량 파일이 없습니다: {RAINFALL_PATH}")

rain_header_idx = None
if RAINFALL_PATH.exists():
    with open(RAINFALL_PATH, encoding="cp949") as f:
        for i, line in enumerate(f):
            if "날짜" in line and "강수량" in line:
                rain_header_idx = i
                break

print(f"\n강수량 헤더 행 인덱스: {rain_header_idx}")

In [ ]:
if RAINFALL_PATH.exists() and rain_header_idx is not None:
    rain_df = pd.read_csv(RAINFALL_PATH, encoding="cp949", skiprows=rain_header_idx)
else:
    rain_df = pd.DataFrame(columns=["날짜", "지점", "강수량(mm)"])

print("=== 강수량 raw ===")
print(rain_df.shape)
print(rain_df.columns)
display(rain_df.head())

date_col = first_existing(rain_df.columns, ["날짜", "일시", "DATE", "date"])
rain_col = first_existing(rain_df.columns, ["강수량(mm)", "강수량", "일강수량(mm)", "rainfall"])

if date_col is None or rain_col is None:
    raise ValueError("강수량 CSV에서 날짜 또는 강수량 컬럼을 찾지 못했습니다.")

rain = rain_df.copy()
rain["날짜"] = pd.to_datetime(rain[date_col], errors="coerce")
rain["기준연도"] = rain["날짜"].dt.year
rain["강수량_mm"] = to_numeric_safe(rain[rain_col]).fillna(0)

rain_yearly = (
    rain.dropna(subset=["기준연도"])
    .groupby("기준연도", as_index=False)
    .agg(
        rain_days=("강수량_mm", lambda s: int((s > 0).sum())),
        heavy_rain_days=("강수량_mm", lambda s: int((s >= 30).sum())),
        total_rainfall=("강수량_mm", "sum"),
        avg_rainfall=("강수량_mm", "mean"),
        max_daily_rainfall=("강수량_mm", "max"),
    )
)
rain_yearly["기준연도"] = rain_yearly["기준연도"].astype(int)
rain_yearly = rain_yearly[rain_yearly["기준연도"].isin(TARGET_YEARS)].copy()

print("=== 연도별 강수량 집계 ===")
print(rain_yearly.shape)
print(rain_yearly.columns)
display(rain_yearly)

---
## 5. TRACE 침수흔적 데이터 전처리

In [ ]:
trace_gdf = None
trace_files = sorted(FLOOD_TRACE_DIR.rglob("서울시_*.shp")) if FLOOD_TRACE_DIR.exists() else []

print("=== 침수흔적 SHP 후보 ===")
for p in trace_files:
    print(p)

if not trace_files:
    raise FileNotFoundError("TRACE/침수흔적 SHP 파일을 찾지 못했습니다.")

trace_list = []
for shp in trace_files:
    tmp = gpd.read_file(shp)
    source_year = None
    for token in [shp.stem, shp.parent.name]:
        digits = "".join(ch for ch in token if ch.isdigit())
        if len(digits) >= 4:
            source_year = int(digits[:4])
            break

    year_col = first_existing(tmp.columns, ["F_YR", "INV_YR", "YEAR", "year", "기준연도"])
    if year_col:
        tmp["기준연도"] = pd.to_numeric(tmp[year_col], errors="coerce")
    else:
        tmp["기준연도"] = source_year
    tmp["기준연도"] = tmp["기준연도"].fillna(source_year).astype("Int64")
    tmp["source_file"] = shp.name
    trace_list.append(tmp)
    print(f"  → {shp.name} 읽기 성공: {tmp.shape}")

trace_gdf = pd.concat(trace_list, ignore_index=True)
trace_gdf = gpd.GeoDataFrame(trace_gdf, geometry="geometry", crs=trace_list[0].crs)
print_gdf_diagnostics(trace_gdf, "침수흔적 원본")
trace_gdf = ensure_projected_crs(trace_gdf, source_name="침수흔적도")
print_gdf_diagnostics(trace_gdf, "침수흔적 EPSG:5179")
display(trace_gdf.head())

In [ ]:
# 면적 컬럼은 F_AREA를 우선 사용하고, 없으면 geometry 면적을 보조로 사용합니다.
area_col = first_existing(trace_gdf.columns, ["F_AREA", "f_area", "AREA", "area"])
if area_col:
    trace_gdf["trace_area_value"] = to_numeric_safe(trace_gdf[area_col])
    if trace_gdf["trace_area_value"].isna().all():
        trace_gdf["trace_area_value"] = trace_gdf.geometry.area
        print("[주의] F_AREA 값이 모두 결측이라 geometry.area를 사용합니다.")
else:
    trace_gdf["trace_area_value"] = trace_gdf.geometry.area
    print("[주의] F_AREA 컬럼이 없어 geometry.area를 사용합니다.")

# 침수심은 F_SHIM을 우선, 없으면 F_AVR_HGT를 사용합니다.
depth_col = first_existing(trace_gdf.columns, ["F_SHIM", "F_AVR_HGT", "f_shim", "f_avr_hgt", "DEPTH", "depth"])
if depth_col:
    trace_gdf["trace_depth_value"] = to_numeric_safe(trace_gdf[depth_col])
else:
    trace_gdf["trace_depth_value"] = np.nan
    print("[주의] 침수심 컬럼을 찾지 못해 깊이 변수는 0으로 처리합니다.")

trace_gdf["trace_area_value"] = trace_gdf["trace_area_value"].fillna(0)
trace_gdf["trace_depth_value"] = trace_gdf["trace_depth_value"].fillna(0)

print("침수흔적 사용 컬럼:", {"area_col": area_col, "depth_col": depth_col})
display(trace_gdf[["기준연도", "trace_area_value", "trace_depth_value", "geometry"]].head())

---
## 6. 홍수 예측 데이터 전처리

In [ ]:
pred_files = [FLOOD_PRED_DIR / f"DS_FLOODING_{i}.shp" for i in range(1, 7)]

print("=== 홍수 예측 SHP 후보 ===")
for p in pred_files:
    mark = "✓" if p.exists() else "✗ 없음"
    print(f"[{mark}] {p}")

pred_list = []
for level, shp in enumerate(pred_files, start=1):
    if not shp.exists():
        continue
    tmp = gpd.read_file(shp)
    # DS_FLOODING_1 → 1등급, ..., DS_FLOODING_6 → 6등급으로 임시 해석합니다.
    # 실제 메타데이터가 다르면 메타데이터 기준으로 수정하세요.
    tmp["flood_pred_level"] = level
    tmp["source_file"] = shp.name
    tmp = ensure_projected_crs(tmp, source_name=shp.name)
    pred_list.append(tmp)
    print(f"  → {shp.name} 읽기 성공: {tmp.shape}, level={level}")

if not pred_list:
    raise FileNotFoundError("DS_FLOODING_1~6 SHP 파일을 찾지 못했습니다.")

flood_pred_gdf = pd.concat(pred_list, ignore_index=True)
flood_pred_gdf = gpd.GeoDataFrame(flood_pred_gdf, geometry="geometry", crs=pred_list[0].crs)
flood_pred_gdf = ensure_projected_crs(flood_pred_gdf, source_name="홍수 예측 통합")

print_gdf_diagnostics(flood_pred_gdf, "홍수 예측 EPSG:5179")
display(flood_pred_gdf.head())

---
## 7. 행정동 경계 데이터 불러오기

행정동 경계 파일은 필수입니다. 없으면 IVI 행정동 목록으로 대체하지 않고 중단합니다.

In [ ]:
def find_admin_boundary_file(boundary_dirs):
    patterns = ["*.shp", "*.gpkg", "*.geojson", "*.json"]
    candidates = []
    for directory in boundary_dirs:
        if not directory.exists():
            continue
        for pattern in patterns:
            candidates.extend(directory.rglob(pattern))
    return sorted(candidates)


if ADMIN_BOUNDARY_PATH is not None:
    admin_boundary_files = [Path(ADMIN_BOUNDARY_PATH)]
else:
    admin_boundary_files = find_admin_boundary_file(ADMIN_BOUNDARY_DIRS)

print("=== 행정동 경계 파일 후보 ===")
for p in admin_boundary_files:
    print(p)

if not admin_boundary_files:
    raise FileNotFoundError(
        "행정동 경계 SHP/GPKG/GeoJSON 파일을 찾지 못했습니다. "
        "RII는 공간 결합이 핵심이므로 행정동 경계가 반드시 필요합니다. "
        "data/external/seoul_admin_dong_boundary/ 또는 data/raw/boundary/에 경계 파일을 넣거나 "
        "ADMIN_BOUNDARY_PATH에 직접 지정하세요."
    )

admin_path = admin_boundary_files[0]
admin_gdf = gpd.read_file(admin_path)
print_gdf_diagnostics(admin_gdf, "행정동 경계 원본")

admin_gdf = standardize_admin_columns(
    admin_gdf,
    code_col=ADMIN_CODE_COL,
    gu_col=ADMIN_GU_COL,
    dong_col=ADMIN_DONG_COL,
)
admin_gdf = ensure_projected_crs(admin_gdf, source_name="행정동 경계")
admin_gdf = admin_gdf[~admin_gdf.geometry.isna() & ~admin_gdf.geometry.is_empty].copy()

print(f"사용한 행정동 경계 파일: {admin_path}")
print_gdf_diagnostics(admin_gdf, "행정동 경계 EPSG:5179")
display(admin_gdf.head())

admin_attrs = admin_gdf[[c for c in ["행정동코드", "자치구명", "행정동명", "dong_key"] if c in admin_gdf.columns]].drop_duplicates().copy()
admin_template = []
for year in TARGET_YEARS:
    tmp = admin_attrs.copy()
    tmp["기준연도"] = year
    admin_template.append(tmp)
admin_template = pd.concat(admin_template, ignore_index=True)

print("=== 행정동 템플릿 ===")
print(admin_template.shape)
display(admin_template.head())

---
## 8. 공간 결합

기본 결합은 `intersects` 기준입니다. 결합 0건이면 bounds와 CRS 진단을 출력하고 중단합니다.

In [ ]:
def spatial_join_to_admin(source_gdf, admin_gdf, source_name="source"):
    if source_gdf is None or source_gdf.empty:
        raise ValueError(f"{source_name}: 원천 공간 데이터가 없습니다.")
    if admin_gdf is None or admin_gdf.empty:
        raise ValueError(f"{source_name}: 행정동 경계가 없습니다.")

    print_gdf_diagnostics(source_gdf, f"{source_name} 결합 전")
    print_gdf_diagnostics(admin_gdf, "행정동 경계 결합 전")

    source_gdf = ensure_projected_crs(source_gdf, source_name=source_name)
    admin_gdf = ensure_projected_crs(admin_gdf, source_name="행정동 경계")
    source_gdf = source_gdf[~source_gdf.geometry.isna() & ~source_gdf.geometry.is_empty].copy()
    admin_gdf = admin_gdf[~admin_gdf.geometry.isna() & ~admin_gdf.geometry.is_empty].copy()

    # 손상 geometry가 있으면 buffer(0)로 보정합니다.
    source_gdf["geometry"] = source_gdf.geometry.buffer(0)
    admin_gdf["geometry"] = admin_gdf.geometry.buffer(0)

    print_gdf_diagnostics(source_gdf, f"{source_name} EPSG:5179")
    print_gdf_diagnostics(admin_gdf, "행정동 경계 EPSG:5179")
    print("bounds 겹침 여부:", bounds_overlap(source_gdf, admin_gdf))

    admin_cols = [c for c in ["행정동코드", "자치구명", "행정동명", "dong_key", "geometry"] if c in admin_gdf.columns]
    joined = gpd.sjoin(source_gdf, admin_gdf[admin_cols], how="left", predicate="intersects")

    match_count = int(joined["dong_key"].notna().sum()) if "dong_key" in joined.columns else 0
    print(f"{source_name} 공간 결합 결과: 전체 {len(joined):,}건 / 행정동 매칭 {match_count:,}건")

    if match_count == 0:
        diagnose_join_failure(source_gdf, admin_gdf, source_name)
        raise ValueError(f"{source_name} 공간 결합 매칭이 0건입니다. CRS 또는 행정동 경계 파일을 확인하세요.")
    if match_count < len(joined):
        print(f"[주의] {source_name}: {len(joined) - match_count:,}건은 어떤 행정동과도 매칭되지 않았습니다.")

    display(joined[[c for c in ["기준연도", "행정동코드", "자치구명", "행정동명", "dong_key"] if c in joined.columns]].head())
    return joined


trace_joined = spatial_join_to_admin(trace_gdf, admin_gdf, source_name="침수흔적")
pred_joined = spatial_join_to_admin(flood_pred_gdf, admin_gdf, source_name="홍수 예측")

---
## 9. 행정동 단위 집계

In [ ]:
def aggregate_trace_by_dong(trace_joined):
    group_cols = [c for c in ["기준연도", "행정동코드", "자치구명", "행정동명", "dong_key"] if c in trace_joined.columns]
    valid = trace_joined.dropna(subset=["dong_key"]).copy()

    agg = (
        valid.groupby(group_cols, dropna=False)
        .agg(
            flood_trace_count=("geometry", "count"),
            flood_trace_area=("trace_area_value", "sum"),
            avg_flood_depth=("trace_depth_value", "mean"),
            max_flood_depth=("trace_depth_value", "max"),
        )
        .reset_index()
    )
    agg["flood_trace_flag"] = (agg["flood_trace_count"] > 0).astype(int)
    return agg


def aggregate_pred_by_dong(pred_joined):
    group_cols = [c for c in ["행정동코드", "자치구명", "행정동명", "dong_key"] if c in pred_joined.columns]
    valid = pred_joined.dropna(subset=["dong_key"]).copy()
    valid["flood_pred_level"] = pd.to_numeric(valid["flood_pred_level"], errors="coerce").fillna(0)

    agg = (
        valid.groupby(group_cols, dropna=False)
        .agg(
            flood_pred_count=("geometry", "count"),
            flood_pred_level_max=("flood_pred_level", "max"),
            flood_pred_level_mean=("flood_pred_level", "mean"),
            flood_pred_weighted_score=("flood_pred_level", "sum"),
        )
        .reset_index()
    )
    agg["flood_pred_flag"] = (agg["flood_pred_count"] > 0).astype(int)
    return agg


trace_agg_event = aggregate_trace_by_dong(trace_joined)
pred_agg_base = aggregate_pred_by_dong(pred_joined)

# TRACE 원자료는 2017~2020처럼 기준연도보다 과거인 경우가 많습니다.
# RII의 침수흔적은 "과거 침수 이력"이므로 각 기준연도 이하의 누적 이력으로 반영합니다.
trace_agg = []
trace_key_cols = [c for c in ["행정동코드", "자치구명", "행정동명", "dong_key"] if c in trace_agg_event.columns]
for year in TARGET_YEARS:
    hist = trace_agg_event[trace_agg_event["기준연도"] <= year].copy()
    if hist.empty:
        continue
    tmp = (
        hist.groupby(trace_key_cols, dropna=False)
        .agg(
            flood_trace_count=("flood_trace_count", "sum"),
            flood_trace_area=("flood_trace_area", "sum"),
            avg_flood_depth=("avg_flood_depth", "mean"),
            max_flood_depth=("max_flood_depth", "max"),
            flood_trace_flag=("flood_trace_flag", "max"),
        )
        .reset_index()
    )
    tmp["기준연도"] = year
    trace_agg.append(tmp)
trace_agg = pd.concat(trace_agg, ignore_index=True) if trace_agg else trace_agg_event.iloc[0:0].copy()

# 홍수 예측 데이터는 특정 연도 컬럼이 없으므로 구조적 위험으로 보고 모든 기준연도에 동일 적용합니다.
pred_agg = []
for year in TARGET_YEARS:
    tmp = pred_agg_base.copy()
    tmp["기준연도"] = year
    pred_agg.append(tmp)
pred_agg = pd.concat(pred_agg, ignore_index=True)

print("=== 침수흔적 행정동 집계 ===")
print(trace_agg.shape)
display(trace_agg.head())

print("=== 홍수 예측 행정동 집계 ===")
print(pred_agg.shape)
display(pred_agg.head())

---
## 10. RII 산출

In [ ]:
rii_df = admin_template.copy()
rii_df = rii_df.merge(rain_yearly, on="기준연도", how="left")

use_trace_code = (
    "행정동코드" in rii_df.columns and "행정동코드" in trace_agg.columns
    and rii_df["행정동코드"].notna().any() and trace_agg["행정동코드"].notna().any()
)
use_pred_code = (
    "행정동코드" in rii_df.columns and "행정동코드" in pred_agg.columns
    and rii_df["행정동코드"].notna().any() and pred_agg["행정동코드"].notna().any()
)

trace_keys = ["기준연도", "행정동코드"] if use_trace_code else ["기준연도", "dong_key"]
pred_keys = ["기준연도", "행정동코드"] if use_pred_code else ["기준연도", "dong_key"]
print("TRACE 병합 키:", trace_keys)
print("홍수예측 병합 키:", pred_keys)

trace_drop = [c for c in ["행정동코드", "자치구명", "행정동명", "dong_key"] if c in trace_agg.columns and c not in trace_keys]
pred_drop = [c for c in ["행정동코드", "자치구명", "행정동명", "dong_key"] if c in pred_agg.columns and c not in pred_keys]

rii_df = rii_df.merge(trace_agg.drop(columns=trace_drop, errors="ignore"), on=trace_keys, how="left")
rii_df = rii_df.merge(pred_agg.drop(columns=pred_drop, errors="ignore"), on=pred_keys, how="left")

fill_zero_cols = [
    "rain_days", "heavy_rain_days", "total_rainfall", "avg_rainfall", "max_daily_rainfall",
    "flood_trace_count", "flood_trace_area", "avg_flood_depth", "max_flood_depth", "flood_trace_flag",
    "flood_pred_count", "flood_pred_level_max", "flood_pred_level_mean", "flood_pred_weighted_score", "flood_pred_flag",
]
for col in fill_zero_cols:
    if col not in rii_df.columns:
        rii_df[col] = 0
    rii_df[col] = pd.to_numeric(rii_df[col], errors="coerce").fillna(0)

rain_components = ["rain_days", "heavy_rain_days", "total_rainfall", "max_daily_rainfall"]
for col in rain_components:
    rii_df[f"{col}_score"] = minmax_scale(rii_df[col])
rii_df["rain_exposure_score"] = rii_df[[f"{col}_score" for col in rain_components]].mean(axis=1)

trace_components = ["flood_trace_count", "flood_trace_area", "avg_flood_depth", "max_flood_depth"]
for col in trace_components:
    rii_df[f"{col}_score"] = minmax_scale(rii_df[col])
rii_df["flood_trace_score"] = rii_df[[f"{col}_score" for col in trace_components]].mean(axis=1)

pred_components = ["flood_pred_count", "flood_pred_level_max", "flood_pred_level_mean", "flood_pred_weighted_score"]
for col in pred_components:
    rii_df[f"{col}_score"] = minmax_scale(rii_df[col])
rii_df["flood_prediction_score"] = rii_df[[f"{col}_score" for col in pred_components]].mean(axis=1)

if rii_df["flood_trace_score"].sum() == 0:
    print("[경고] flood_trace_score가 전부 0입니다. TRACE 공간 결합/면적/침수심 컬럼을 확인하세요.")
if rii_df["flood_prediction_score"].sum() == 0:
    print("[경고] flood_prediction_score가 전부 0입니다. DS_FLOODING 공간 결합/등급 부여를 확인하세요.")

rii_df["RII"] = rii_df[
    ["rain_exposure_score", "flood_trace_score", "flood_prediction_score"]
].mean(axis=1)

# 순위와 등급은 기준연도 내부에서만 계산합니다.
rii_df = calculate_rank_grade_by_year(rii_df, score_col="RII")

for year, nunique in rii_df.groupby("기준연도")["RII"].nunique().items():
    if nunique <= 1:
        print(f"[경고] {year}년 RII 값이 모든 행정동에서 동일합니다. 공간 결합 또는 구성 점수 변별력을 확인하세요.")

print("=== RII 산출 결과 ===")
print(rii_df.shape)
display(rii_df.head())

---
## 11. 최종 저장

In [ ]:
final_cols = [
    "기준연도", "자치구명", "행정동명", "행정동코드",
    "rain_days", "heavy_rain_days", "total_rainfall", "avg_rainfall", "max_daily_rainfall",
    "flood_trace_count", "flood_trace_area", "avg_flood_depth", "max_flood_depth", "flood_trace_flag",
    "flood_pred_count", "flood_pred_level_max", "flood_pred_level_mean", "flood_pred_weighted_score", "flood_pred_flag",
    "rain_exposure_score", "flood_trace_score", "flood_prediction_score",
    "RII", "RII_rank", "RII_grade",
]
final_cols = [c for c in final_cols if c in rii_df.columns]
rii_final = rii_df[final_cols].copy()

rii_final.to_csv(OUT_ALL_CSV, index=False, encoding="utf-8-sig")

# 2021년 파일은 2021년만 필터링한 뒤 rank/grade를 다시 계산합니다.
rii_2021 = rii_final[rii_final["기준연도"] == 2021].copy()
rii_2021 = rii_2021.drop(columns=["RII_rank", "RII_grade"], errors="ignore")
rii_2021 = calculate_rank_grade_by_year(rii_2021, score_col="RII")

if rii_2021["RII"].nunique(dropna=False) <= 1:
    print("[경고] 2021년 RII 값이 모든 행정동에서 동일합니다. TRACE/홍수예측 공간 결합 결과를 확인하세요.")

rii_2021.to_csv(OUT_2021_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUT_ALL_CSV}")
print(f"저장 완료: {OUT_2021_CSV}")

admin_for_map = admin_gdf[[c for c in ["행정동코드", "자치구명", "행정동명", "dong_key", "geometry"] if c in admin_gdf.columns]].copy()
map_df = admin_for_map.merge(
    rii_df[rii_df["기준연도"] == 2021].drop(columns=["자치구명", "행정동명", "행정동코드"], errors="ignore"),
    on="dong_key",
    how="left",
)
map_gdf = gpd.GeoDataFrame(map_df, geometry="geometry", crs=admin_gdf.crs).to_crs(TARGET_CRS)
map_gdf.to_file(OUT_2021_GPKG, layer="rii_by_dong_2021", driver="GPKG")
print(f"저장 완료: {OUT_2021_GPKG}")

---
## 12. 결과 검증

1. 최종 행정동 수가 너무 적지 않은지
2. 행정동명이 중복되어 잘못 병합되지 않았는지
3. RII 값이 0~1 사이인지
4. RII가 높은 지역이 실제로 침수흔적 또는 침수예상 데이터와 연결되는지
5. 2021년 최종 파일이 따로 저장되었는지

In [ ]:
rii_df.head()

In [ ]:
rii_df.info()

In [ ]:
rii_df.isna().sum()

In [ ]:
rii_df.describe()

In [ ]:
rii_df["기준연도"].value_counts().sort_index()

In [ ]:
rii_df["행정동명"].nunique()

In [ ]:
rii_df.sort_values("RII", ascending=False).head(20)

In [ ]:
print("1) 연도별 행정동 수")
display(rii_df.groupby("기준연도")["행정동명"].nunique())

print("\n2) 자치구명+행정동명 중복 여부")
dup_check = (
    rii_df.groupby(["기준연도", "자치구명", "행정동명"], dropna=False)
    .size()
    .reset_index(name="n")
    .query("n > 1")
)
display(dup_check.head(20))

print("\n3) RII 범위")
print(rii_df["RII"].min(), rii_df["RII"].max())
print("RII가 0~1 사이인가?", bool(rii_df["RII"].between(0, 1).all()))

print("\n4) RII 상위 지역의 침수흔적/침수예상 연결 확인")
check_cols = [
    "기준연도", "자치구명", "행정동명", "RII",
    "flood_trace_count", "flood_trace_area", "flood_pred_count", "flood_pred_level_max"
]
display(rii_df.sort_values("RII", ascending=False)[check_cols].head(20))

print("\n5) 2021년 파일 저장 여부")
print(OUT_2021_CSV, OUT_2021_CSV.exists())
print(OUT_2021_GPKG, OUT_2021_GPKG.exists())